# ML Assignment 2 — Classification Models + Streamlit App**BITS Pilani WILP — M.Tech (AIML/DSE)**This notebook trains 5 classifiers on one dataset, computes 6 evaluation metrics foreach, and exports everything the Streamlit app needs.**Run order:** top to bottom. Only Cell 2 (CONFIG) needs editing.

## 1. Setup

In [ ]:
!pip install -q scikit-learn pandas numpy matplotlib seaborn joblibimport os, json, warningsimport numpy as npimport pandas as pdimport joblibimport matplotlib.pyplot as pltimport seaborn as snsfrom sklearn.model_selection import train_test_splitfrom sklearn.pipeline import Pipelinefrom sklearn.compose import ColumnTransformerfrom sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoderfrom sklearn.impute import SimpleImputerfrom sklearn.linear_model import LogisticRegressionfrom sklearn.tree import DecisionTreeClassifierfrom sklearn.neighbors import KNeighborsClassifierfrom sklearn.naive_bayes import GaussianNBfrom sklearn.ensemble import RandomForestClassifierfrom sklearn.metrics import (accuracy_score, roc_auc_score, precision_score,                             recall_score, f1_score, matthews_corrcoef,                             confusion_matrix, classification_report)warnings.filterwarnings("ignore")RANDOM_STATE = 42np.random.seed(RANDOM_STATE)print("Setup complete.")

## 2. CONFIG — edit this cell onlyUpload your dataset CSV using the file browser on the left (folder icon), or run theupload widget below. Then set `CSV_PATH` and `TARGET_COL`.

In [ ]:
# --- Option A: upload from your machine ---from google.colab import filesuploaded = files.upload()          # pick your dataset CSVCSV_PATH = list(uploaded.keys())[0]# --- Option B: if the file is already in Colab, comment out the block above and set: ---# CSV_PATH = "my_dataset.csv"TARGET_COL   = "REPLACE_WITH_YOUR_TARGET_COLUMN"DROP_COLS    = []          # e.g. ["id", "Unnamed: 0"] - identifier columns to discardTEST_SIZE    = 0.2PROJECT_NAME = "ml-assignment-2"print("CSV:", CSV_PATH)

## 3. Load and inspect the data

In [ ]:
df = pd.read_csv(CSV_PATH)df = df.drop(columns=[c for c in DROP_COLS if c in df.columns])print("Shape:", df.shape)print("Features (excluding target):", df.shape[1] - 1, " | Instances:", df.shape[0])assert df.shape[1] - 1 >= 12, "Need at least 12 features - pick a wider dataset."assert df.shape[0] >= 500,   "Need at least 500 instances - pick a bigger dataset."assert TARGET_COL in df.columns, f"'{TARGET_COL}' not found. Columns: {list(df.columns)}"display(df.head())print("\nMissing values per column:")print(df.isna().sum()[df.isna().sum() > 0])

In [ ]:
print("Target distribution:")print(df[TARGET_COL].value_counts())plt.figure(figsize=(6, 4))sns.countplot(x=df[TARGET_COL].astype(str), palette="viridis")plt.title("Class distribution")plt.xticks(rotation=30, ha="right")plt.tight_layout()plt.show()

In [ ]:
# Correlation heatmap of numeric features (useful for the README write-up)num_only = df.select_dtypes(include=np.number)if num_only.shape[1] > 1:    plt.figure(figsize=(10, 8))    sns.heatmap(num_only.corr(), cmap="coolwarm", center=0, square=True, cbar_kws={"shrink": .7})    plt.title("Feature correlation")    plt.tight_layout()    plt.show()

## 4. Split features / target, build preprocessing

In [ ]:
X = df.drop(columns=[TARGET_COL])y_raw = df[TARGET_COL]label_encoder = LabelEncoder()y = label_encoder.fit_transform(y_raw)CLASS_NAMES = [str(c) for c in label_encoder.classes_]N_CLASSES = len(CLASS_NAMES)IS_BINARY = (N_CLASSES == 2)numeric_cols     = X.select_dtypes(include=np.number).columns.tolist()categorical_cols = [c for c in X.columns if c not in numeric_cols]print(f"Task: {'binary' if IS_BINARY else 'multi-class'} ({N_CLASSES} classes)")print("Classes:", CLASS_NAMES)print(f"Numeric features: {len(numeric_cols)} | Categorical features: {len(categorical_cols)}")

In [ ]:
def build_preprocessor():    """Impute + scale numerics, impute + one-hot encode categoricals."""    try:        ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)    except TypeError:                      # older scikit-learn        ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)    numeric_branch = Pipeline([        ("impute", SimpleImputer(strategy="median")),        ("scale",  StandardScaler()),    ])    categorical_branch = Pipeline([        ("impute", SimpleImputer(strategy="most_frequent")),        ("encode", ohe),    ])    return ColumnTransformer([        ("num", numeric_branch, numeric_cols),        ("cat", categorical_branch, categorical_cols),    ])X_train, X_test, y_train, y_test = train_test_split(    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)print("Train:", X_train.shape, " Test:", X_test.shape)

### Export the test split as `test_data.csv`This is the file the Streamlit app will consume, and the CSV that goes in the repo.

In [ ]:
test_export = X_test.copy()test_export[TARGET_COL] = label_encoder.inverse_transform(y_test)test_export.to_csv("test_data.csv", index=False)print("Wrote test_data.csv:", test_export.shape)display(test_export.head())

## 5. Train the five models

In [ ]:
MODELS = {    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),    "Decision Tree": DecisionTreeClassifier(max_depth=8, min_samples_leaf=5,                                            random_state=RANDOM_STATE),    "kNN": KNeighborsClassifier(n_neighbors=7, weights="distance"),    "Naive Bayes": GaussianNB(),    "Random Forest (Ensemble)": RandomForestClassifier(        n_estimators=300, max_depth=None, min_samples_leaf=2,        random_state=RANDOM_STATE, n_jobs=-1),}FILE_KEYS = {    "Logistic Regression": "logistic_regression",    "Decision Tree": "decision_tree",    "kNN": "knn",    "Naive Bayes": "naive_bayes",    "Random Forest (Ensemble)": "random_forest",}

In [ ]:
def score_model(y_true, y_pred, y_proba):    """All six required metrics. Averaging switches with binary vs multi-class."""    avg = "binary" if IS_BINARY else "weighted"    if IS_BINARY:        auc = roc_auc_score(y_true, y_proba[:, 1])    else:        auc = roc_auc_score(y_true, y_proba, multi_class="ovr", average="weighted")    return {        "Accuracy":  accuracy_score(y_true, y_pred),        "AUC":       auc,        "Precision": precision_score(y_true, y_pred, average=avg, zero_division=0),        "Recall":    recall_score(y_true, y_pred, average=avg, zero_division=0),        "F1":        f1_score(y_true, y_pred, average=avg, zero_division=0),        "MCC":       matthews_corrcoef(y_true, y_pred),    }

In [ ]:
os.makedirs("model", exist_ok=True)results, fitted, predictions = [], {}, {}for name, estimator in MODELS.items():    pipe = Pipeline([("preprocess", build_preprocessor()), ("classifier", estimator)])    pipe.fit(X_train, y_train)    y_pred  = pipe.predict(X_test)    y_proba = pipe.predict_proba(X_test)    results.append({"ML Model Name": name, **score_model(y_test, y_pred, y_proba)})    fitted[name] = pipe    predictions[name] = y_pred    joblib.dump(pipe, f"model/{FILE_KEYS[name]}.joblib")    print(f"trained + saved: {name}")joblib.dump({    "target_col": TARGET_COL,    "class_names": CLASS_NAMES,    "is_binary": IS_BINARY,    "feature_cols": list(X.columns),    "label_classes": list(label_encoder.classes_),}, "model/metadata.joblib")print("\nSaved model/metadata.joblib")

## 6. Comparison table (paste this into your README)

In [ ]:
comparison = pd.DataFrame(results).set_index("ML Model Name").round(4)display(comparison)winner = comparison["F1"].idxmax()print(f"\nBest F1: {winner} ({comparison.loc[winner, 'F1']:.4f})")print(f"Best Accuracy: {comparison['Accuracy'].idxmax()}")print(f"Best MCC: {comparison['MCC'].idxmax()}")comparison.to_csv("model/metrics_comparison.csv")print("\n--- Markdown for README.md ---\n")print(comparison.reset_index().to_markdown(index=False))

## 7. Confusion matrices

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(24, 4.5))for ax, (name, y_pred) in zip(axes, predictions.items()):    cm = confusion_matrix(y_test, y_pred)    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)    ax.set_title(name, fontsize=10)    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")plt.tight_layout()plt.savefig("model/confusion_matrices.png", dpi=120, bbox_inches="tight")plt.show()

In [ ]:
for name, y_pred in predictions.items():    print("=" * 60)    print(name)    print("=" * 60)    print(classification_report(y_test, y_pred, target_names=CLASS_NAMES, zero_division=0))

## 8. Package everything for GitHubThis writes `app.py`, `requirements.txt` and a README skeleton next to the savedmodels, then zips the lot so you can download it in one click.

In [ ]:
requirements = """streamlitscikit-learnpandasnumpymatplotlibseabornjoblib"""with open("requirements.txt", "w") as f:    f.write(requirements)# Pin scikit-learn to the version used here - version drift is a common deploy failure.import sklearnprint("Trained with scikit-learn", sklearn.__version__)print("Consider pinning: scikit-learn==" + sklearn.__version__)

In [ ]:
!zip -rq submission_bundle.zip model test_data.csv requirements.txtfrom google.colab import filesfiles.download("submission_bundle.zip")

---### What to do next1. Download `submission_bundle.zip` and unzip it.2. Drop `app.py` (provided separately) into the same folder.3. Push the folder to a new GitHub repo.4. Deploy on Streamlit Community Cloud.5. Re-run this notebook once on BITS Virtual Lab and screenshot it.